# HELOC Stage 08: Cross-method XAI evaluation

This notebook evaluates the persisted SHAP, LIME, and DiCE results for the fixed HELOC cohort. It does not regenerate explanations or alter the frozen model workflow.

## 1. Artifact inventory

All required Stage 04–07 artifacts must exist before evaluation begins. Missing evidence is not regenerated.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns",None); pd.set_option("display.width",180)
plt.style.use("seaborn-v0_8-whitegrid")
PROJECT_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"backend/artifacts/heloc/xai_evaluation_cases.csv").is_file()),None)
if PROJECT_ROOT is None: raise FileNotFoundError("Could not locate project root")
ARTIFACT_DIR=PROJECT_ROOT/"backend/artifacts/heloc"; PLOT_DIR=ARTIFACT_DIR/"plots"; PLOT_DIR.mkdir(exist_ok=True)

required={
"xai_evaluation_cases.csv":"Fixed XAI cohort","model_metadata.json":"Frozen model metadata","preprocessing_metadata.json":"Frozen preprocessing metadata",
"shap_global_importance.csv":"SHAP global importance","shap_local_explanations.csv":"SHAP local records","shap_top_factors.csv":"SHAP top-five original features","shap_sparsity_metrics.csv":"SHAP compactness","shap_preliminary_stability.csv":"SHAP perturbation diagnostic","shap_preliminary_faithfulness.csv":"SHAP sensitivity diagnostic","shap_metadata.json":"SHAP provenance",
"lime_local_explanations.csv":"LIME local records","lime_top_factors.csv":"LIME top-five original features","lime_sparsity_metrics.csv":"LIME compactness","lime_preliminary_stability.csv":"LIME same-seed diagnostic","lime_preliminary_faithfulness.csv":"LIME sensitivity diagnostic","lime_local_fidelity.csv":"LIME local fidelity","lime_metadata.json":"LIME provenance",
"dice_counterfactuals.csv":"Validated DiCE counterfactuals","dice_feature_changes.csv":"DiCE feature changes","dice_sparsity_metrics.csv":"DiCE sparsity","dice_proximity_metrics.csv":"DiCE proximity","dice_plain_english.csv":"DiCE narratives","dice_generation_first_pass.csv":"Historical first pass","dice_technical_retry.csv":"Controlled technical retry","dice_final_case_status.csv":"Final DiCE statuses","dice_case_summary.csv":"Final case summary","dice_metadata.json":"DiCE provenance"}
inventory=[]
for name,purpose in required.items():
    path=ARTIFACT_DIR/name; rows=np.nan
    if path.exists() and path.suffix==".csv": rows=len(pd.read_csv(path))
    inventory.append({"artifact":name,"exists":path.exists(),"rows":rows,"purpose":purpose})
artifact_inventory=pd.DataFrame(inventory)
assert artifact_inventory.exists.all(), artifact_inventory.loc[~artifact_inventory.exists,"artifact"].tolist()
artifact_inventory

,artifact,exists,rows,purpose
0,xai_evaluation_cases.csv,True,20.0,Fixed XAI cohort
1,model_metadata.json,True,NaN,Frozen model metadata
2,preprocessing_metadata.json,True,NaN,Frozen preprocessing metadata
3,shap_global_importance.csv,True,22.0,SHAP global importance
4,shap_local_explanations.csv,True,440.0,SHAP local records
5,shap_top_factors.csv,True,100.0,SHAP top-five original features
6,shap_sparsity_metrics.csv,True,20.0,SHAP compactness
7,shap_preliminary_stability.csv,True,5.0,SHAP perturbation diagnostic
8,shap_preliminary_faithfulness.csv,True,5.0,SHAP sensitivity diagnostic
9,shap_metadata.json,True,NaN,SHAP provenance


## 2. Common cohort and persisted evidence

SHAP and LIME are compared at the original 22-feature level. DiCE uses the final status after the controlled technical retry.

In [2]:
cases=pd.read_csv(ARTIFACT_DIR/"xai_evaluation_cases.csv")
shap_top=pd.read_csv(ARTIFACT_DIR/"shap_top_factors.csv"); lime_top=pd.read_csv(ARTIFACT_DIR/"lime_top_factors.csv")
shap_sparse=pd.read_csv(ARTIFACT_DIR/"shap_sparsity_metrics.csv"); lime_sparse=pd.read_csv(ARTIFACT_DIR/"lime_sparsity_metrics.csv")
shap_stability=pd.read_csv(ARTIFACT_DIR/"shap_preliminary_stability.csv"); lime_stability=pd.read_csv(ARTIFACT_DIR/"lime_preliminary_stability.csv")
shap_faith=pd.read_csv(ARTIFACT_DIR/"shap_preliminary_faithfulness.csv"); lime_faith=pd.read_csv(ARTIFACT_DIR/"lime_preliminary_faithfulness.csv")
lime_fidelity=pd.read_csv(ARTIFACT_DIR/"lime_local_fidelity.csv")
dice_cf=pd.read_csv(ARTIFACT_DIR/"dice_counterfactuals.csv"); dice_changes=pd.read_csv(ARTIFACT_DIR/"dice_feature_changes.csv")
dice_sparse=pd.read_csv(ARTIFACT_DIR/"dice_sparsity_metrics.csv"); dice_proximity=pd.read_csv(ARTIFACT_DIR/"dice_proximity_metrics.csv")
dice_first=pd.read_csv(ARTIFACT_DIR/"dice_generation_first_pass.csv"); dice_technical=pd.read_csv(ARTIFACT_DIR/"dice_technical_retry.csv")
dice_status=pd.read_csv(ARTIFACT_DIR/"dice_final_case_status.csv"); dice_summary=pd.read_csv(ARTIFACT_DIR/"dice_case_summary.csv")
assert len(cases)==20 and cases.case_id.is_unique and cases.case_type.value_counts().eq(5).all()
assert shap_top.case_id.nunique()==20 and lime_top.case_id.nunique()==20 and dice_status.case_id.nunique()==20
assert len(shap_top)==100 and len(lime_top)==100
print("Common cases verified:",len(cases))

Common cases verified: 20


## 3. SHAP–LIME feature and direction agreement

Agreement concerns feature identity and direction, not attribution magnitude. Rank correlation is left missing when fewer than two shared features make it undefined.

In [3]:
agreement=[]; directional=[]
for case_id in cases.case_id:
    s=shap_top[shap_top.case_id.eq(case_id)].sort_values("rank"); l=lime_top[lime_top.case_id.eq(case_id)].sort_values("rank")
    sf=s.original_feature.tolist(); lf=l.original_feature.tolist(); shared=set(sf)&set(lf)
    sr=pd.Series({f:i+1 for i,f in enumerate(sf)}); lr=pd.Series({f:i+1 for i,f in enumerate(lf)})
    rank_corr=sr.loc[list(shared)].corr(lr.loc[list(shared)],method="spearman") if len(shared)>=2 else np.nan
    agreement.append({"case_id":case_id,"top_1_agreement":sf[0]==lf[0],"top_3_overlap_count":len(set(sf[:3])&set(lf[:3])),
        "top_3_jaccard":len(set(sf[:3])&set(lf[:3]))/len(set(sf[:3])|set(lf[:3])),"top_5_overlap_count":len(shared),
        "top_5_jaccard":len(shared)/len(set(sf)|set(lf)),"shared_top_5_rank_spearman":rank_corr})
    sd=s.set_index("original_feature").direction; ld=l.set_index("original_feature").direction
    canonical_direction=lambda value: "positive" if str(value).lower().startswith("increases") else "negative" if str(value).lower().startswith("decreases") else np.nan
    same=sum(canonical_direction(sd[f])==canonical_direction(ld[f]) for f in shared)
    directional.append({"case_id":case_id,"shared_feature_count":len(shared),"same_direction_count":same,"direction_agreement_rate":same/len(shared) if shared else np.nan})
agreement=pd.DataFrame(agreement); direction=pd.DataFrame(directional)
agreement.to_csv(ARTIFACT_DIR/"xai_shap_lime_agreement.csv",index=False); direction.to_csv(ARTIFACT_DIR/"xai_direction_agreement.csv",index=False)
overall_direction=direction.same_direction_count.sum()/direction.shared_feature_count.sum()
print("Top-1 agreements:",int(agreement.top_1_agreement.sum()))
print("Mean top-3 overlap:",agreement.top_3_overlap_count.mean()); print("Mean top-3 Jaccard:",agreement.top_3_jaccard.mean())
print("Mean top-5 overlap:",agreement.top_5_overlap_count.mean()); print("Mean top-5 Jaccard:",agreement.top_5_jaccard.mean())
print("Overall direction agreement:",overall_direction)

Top-1 agreements: 8
Mean top-3 overlap: 2.1
Mean top-3 Jaccard: 0.5599999999999999
Mean top-5 overlap: 3.9
Mean top-5 Jaccard: 0.6523809523809523
Overall direction agreement: 0.9487179487179487


## 4. Sparsity and compactness

DiCE counts counterfactual-modifiable financial-state variables, not directly actionable features. Lower counts are descriptive and do not automatically indicate a better explanation.

In [4]:
def stat_rows(method,metric,values):
    v=pd.Series(values).dropna(); return {"method":method,"metric":metric,"count":len(v),"mean":v.mean(),"median":v.median(),"min":v.min(),"max":v.max()}
sparsity_comparison=pd.DataFrame([
    stat_rows("SHAP","features_for_50_percent",shap_sparse.features_for_50_percent),stat_rows("SHAP","features_for_80_percent",shap_sparse.features_for_80_percent),
    stat_rows("LIME","features_for_50_percent",lime_sparse.features_for_50_percent),stat_rows("LIME","features_for_80_percent",lime_sparse.features_for_80_percent),
    stat_rows("DiCE","changed counterfactual-modifiable financial-state variables",dice_sparse.number_of_changed_modifiable_features)])
sparsity_comparison.to_csv(ARTIFACT_DIR/"xai_sparsity_comparison.csv",index=False); sparsity_comparison

,method,metric,count,mean,median,min,max
0,SHAP,features_for_50_percent,20,3.10,3.0,2,4
1,SHAP,features_for_80_percent,20,6.50,6.0,5,9
2,LIME,features_for_50_percent,20,2.25,2.0,2,3
3,LIME,features_for_80_percent,20,4.00,4.0,4,4
4,DiCE,changed counterfactual-modifiable financial-st...,8,2.75,3.0,2,3


## 5. Persisted stability and sensitivity diagnostics

The SHAP diagnostic uses a small deterministic perturbation of revolving burden. The LIME diagnostic tests same-seed reproduction, so these results are reported separately rather than treated as identical stability tests. Sensitivity results are preliminary intervention-based model diagnostics, not causal evidence.

In [5]:
lime_overlap=np.where(lime_stability.same_top_5_features.astype(bool),5,np.nan)
stability_comparison=pd.DataFrame([
    {"method":"SHAP","diagnostic_type":"small deterministic local perturbation of revolving burden","cases_tested":len(shap_stability),
     "mean_top_5_overlap":shap_stability.top_5_overlap_count.mean(),"median_top_5_overlap":shap_stability.top_5_overlap_count.median(),"minimum_top_5_overlap":shap_stability.top_5_overlap_count.min(),"maximum_top_5_overlap":shap_stability.top_5_overlap_count.max(),"exact_reproduction_count":np.nan},
    {"method":"LIME","diagnostic_type":"same-seed reproduction","cases_tested":len(lime_stability),"mean_top_5_overlap":np.nanmean(lime_overlap),
     "median_top_5_overlap":np.nanmedian(lime_overlap),"minimum_top_5_overlap":np.nanmin(lime_overlap),"maximum_top_5_overlap":np.nanmax(lime_overlap),"exact_reproduction_count":int(lime_stability.exact_reproduction.sum())}])
stability_comparison.to_csv(ARTIFACT_DIR/"xai_stability_comparison.csv",index=False)
faithfulness_comparison=pd.DataFrame([
    {"method":"SHAP","diagnostic":"preliminary intervention-based model sensitivity diagnostic","cases_tested":len(shap_faith),"mean_absolute_score_change":shap_faith.absolute_score_change.mean(),"median_absolute_score_change":shap_faith.absolute_score_change.median(),"minimum_absolute_score_change":shap_faith.absolute_score_change.min(),"maximum_absolute_score_change":shap_faith.absolute_score_change.max(),"interventions_selected_independently":True},
    {"method":"LIME","diagnostic":"preliminary intervention-based model sensitivity diagnostic","cases_tested":len(lime_faith),"mean_absolute_score_change":lime_faith.absolute_score_change.mean(),"median_absolute_score_change":lime_faith.absolute_score_change.median(),"minimum_absolute_score_change":lime_faith.absolute_score_change.min(),"maximum_absolute_score_change":lime_faith.absolute_score_change.max(),"interventions_selected_independently":True}])
faithfulness_comparison.to_csv(ARTIFACT_DIR/"xai_faithfulness_comparison.csv",index=False)
display(stability_comparison); display(faithfulness_comparison)

,method,diagnostic_type,cases_tested,mean_top_5_overlap,median_top_5_overlap,minimum_top_5_overlap,maximum_top_5_overlap,exact_reproduction_count
0,SHAP,small deterministic local perturbation of revo...,5,5.0,5.0,5.0,5.0,NaN
1,LIME,same-seed reproduction,5,5.0,5.0,5.0,5.0,5.0


,method,diagnostic,cases_tested,mean_absolute_score_change,median_absolute_score_change,minimum_absolute_score_change,maximum_absolute_score_change,interventions_selected_independently
0,SHAP,preliminary intervention-based model sensitivi...,5,0.077449,0.075667,0.052755,0.118644,True
1,LIME,preliminary intervention-based model sensitivi...,5,0.047352,0.025823,0.001642,0.118644,True


## 6. LIME local surrogate fidelity

The persisted fidelity values are summarized without retuning. Moderate and weak local fits remain visible.

In [6]:
fidelity_summary=pd.DataFrame([stat_rows("LIME","local_fidelity_r2",lime_fidelity.local_fidelity_r2),stat_rows("LIME","absolute_local_prediction_error",lime_fidelity.absolute_local_prediction_error)])
fidelity_summary.to_csv(ARTIFACT_DIR/"xai_lime_local_fidelity_summary.csv",index=False); fidelity_summary

,method,metric,count,mean,median,min,max
0,LIME,local_fidelity_r2,20,0.416343,0.429768,0.160894,0.573275
1,LIME,absolute_local_prediction_error,20,0.104207,0.063888,0.001754,0.297248


## 7. Final DiCE outcome evaluation

DiCE varies three counterfactual-modifiable financial-state variables. Their inclusion does not imply immediate applicant control or guarantee a real-world outcome.

In [7]:
dice_eval=dice_cf[["case_id","counterfactual_id","original_probability","counterfactual_probability","original_class","counterfactual_class","desired_class","valid_counterfactual","training_domain_valid","special_states_preserved","only_eligible_features_changed"]].copy()
dice_eval["absolute_probability_change"]=(dice_eval.counterfactual_probability-dice_eval.original_probability).abs()
dice_eval=dice_eval.merge(dice_sparse,on=["case_id","counterfactual_id"],how="left").merge(dice_proximity[["case_id","counterfactual_id","normalized_proximity_score"]],on=["case_id","counterfactual_id"],how="left")
dice_eval=dice_eval.rename(columns={"number_of_changed_modifiable_features":"number_of_changed_modifiable_features","normalized_proximity_score":"proximity_score","training_domain_valid":"domain_constraints_passed","special_states_preserved":"special_codes_preserved","only_eligible_features_changed":"modifiable_feature_constraint_passed"})
dice_eval.to_csv(ARTIFACT_DIR/"xai_dice_evaluation.csv",index=False)
final_no_cf=int(dice_status.final_status.str.contains("no_counterfactual_returned").sum()); final_technical=int(dice_status.final_status.str.contains("technical_error").sum()); final_timeout=int(dice_status.final_status.str.contains("timeout").sum())
assert len(dice_eval)==8 and final_no_cf==12 and final_technical==0 and final_timeout==0
print("Availability:",len(dice_eval),"/",len(cases),"=",len(dice_eval)/len(cases))
print("Successful directions:",dice_eval.groupby(["original_class","counterfactual_class"]).size().to_dict())
display(dice_eval.describe())

Availability: 8 / 20 = 0.4


Successful directions: {(0, 1): 5, (1, 0): 3}


,original_probability,counterfactual_probability,original_class,counterfactual_class,desired_class,absolute_probability_change,number_of_changed_modifiable_features,proximity_score
count,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.00000,8.000000
mean,0.500208,0.517134,0.375000,0.625000,0.625000,0.062295,2.75000,0.133857
std,0.002513,0.080681,0.517549,0.517549,0.517549,0.052010,0.46291,0.066007
min,0.497784,0.359443,0.000000,0.000000,0.000000,0.000890,2.00000,0.044514
25%,0.498412,0.484384,0.000000,0.000000,0.000000,0.018777,2.75000,0.093591
50%,0.499446,0.522446,0.000000,1.000000,1.000000,0.056470,3.00000,0.127599
75%,0.501066,0.567331,1.000000,1.000000,1.000000,0.089225,3.00000,0.164670
max,0.504185,0.625801,1.000000,1.000000,1.000000,0.144555,3.00000,0.257009


## 8. DiCE technical-retry disclosure

Four first-pass cases raised `ValueError: empty range for randrange()`. The audit traced this to a one-member deduplicated genetic population, where DiCE calculated `top_half = 0` and called `random.randrange(0)`. A sole-parent guard preserved the model, cases, desired classes, three permitted variables, domains, special-code policy, and genetic method. All four controlled retries completed normally and returned no counterfactuals; the historical first-pass record remains preserved.

## 9. Cross-method case summary

The case table retains canonical original-feature names. Transformed special indicators are not treated as separate explanatory features.

In [8]:
top_s=shap_top.sort_values("rank").groupby("case_id").first()[["original_feature","direction"]].rename(columns={"original_feature":"shap_top_feature","direction":"shap_top_direction"})
top_l=lime_top.sort_values("rank").groupby("case_id").first()[["original_feature","direction"]].rename(columns={"original_feature":"lime_top_feature","direction":"lime_top_direction"})
cross=cases.merge(top_s,on="case_id").merge(top_l,on="case_id").merge(agreement[["case_id","top_1_agreement","top_5_overlap_count","top_5_jaccard"]],on="case_id").merge(direction[["case_id","direction_agreement_rate"]],on="case_id")
cross=cross.merge(dice_status[["case_id","valid_counterfactual_available","final_status"]],on="case_id",how="left").rename(columns={"valid_counterfactual_available":"dice_valid_counterfactual","final_status":"dice_final_status"})
cross=cross.merge(dice_eval[["case_id","counterfactual_probability","number_of_changed_modifiable_features"]],on="case_id",how="left").rename(columns={"counterfactual_probability":"dice_counterfactual_probability","number_of_changed_modifiable_features":"dice_changed_modifiable_features"})
cross.to_csv(ARTIFACT_DIR/"xai_cross_method_case_summary.csv",index=False)
assert len(cross)==20 and cross.case_id.is_unique
cross

,case_id,row_index,true_target,predicted_class,predicted_probability,case_type,shap_top_feature,shap_top_direction,lime_top_feature,lime_top_direction,top_1_agreement,top_5_overlap_count,top_5_jaccard,direction_agreement_rate,dice_valid_counterfactual,dice_final_status,dice_counterfactual_probability,dice_changed_modifiable_features
0,HELOC_XAI_001,1991,1,1,0.948322,high-confidence positive,average_duration_of_resolution,increases model at-risk output,average_duration_of_resolution,increases local at-risk prediction,True,3,0.428571,1.00,False,no_counterfactual_returned_first_pass,NaN,NaN
1,HELOC_XAI_002,9808,1,1,0.945186,high-confidence positive,average_duration_of_resolution,increases model at-risk output,net_fraction_of_revolving_burden,increases local at-risk prediction,False,4,0.666667,1.00,False,no_counterfactual_returned_after_technical_retry,NaN,NaN
2,HELOC_XAI_003,1706,1,1,0.944068,high-confidence positive,average_duration_of_resolution,increases model at-risk output,percentage_of_legal_trades,increases local at-risk prediction,False,4,0.666667,1.00,False,no_counterfactual_returned_after_technical_retry,NaN,NaN
3,HELOC_XAI_004,10074,1,1,0.941455,high-confidence positive,average_duration_of_resolution,increases model at-risk output,average_duration_of_resolution,increases local at-risk prediction,True,3,0.428571,1.00,False,no_counterfactual_returned_after_technical_retry,NaN,NaN
4,HELOC_XAI_005,7308,1,1,0.941175,high-confidence positive,average_duration_of_resolution,increases model at-risk output,net_fraction_of_revolving_burden,increases local at-risk prediction,False,4,0.666667,1.00,False,no_counterfactual_returned_after_technical_retry,NaN,NaN
5,HELOC_XAI_006,2151,0,1,0.500089,borderline positive,net_fraction_of_revolving_burden,increases model at-risk output,net_fraction_of_revolving_burden,increases local at-risk prediction,True,4,0.666667,1.00,True,success_first_pass,0.482261,3.0
6,HELOC_XAI_007,2900,1,1,0.501511,borderline positive,average_duration_of_resolution,increases model at-risk output,percentage_of_legal_trades,decreases local at-risk prediction,False,4,0.666667,1.00,False,no_counterfactual_returned_first_pass,NaN,NaN
7,HELOC_XAI_008,5033,0,1,0.503999,borderline positive,net_fraction_of_revolving_burden,increases model at-risk output,net_fraction_of_revolving_burden,increases local at-risk prediction,True,4,0.666667,1.00,True,success_first_pass,0.359443,3.0
8,HELOC_XAI_009,8449,0,1,0.504185,borderline positive,net_fraction_of_revolving_burden,decreases model at-risk output,net_fraction_of_revolving_burden,decreases local at-risk prediction,True,4,0.666667,1.00,True,success_first_pass,0.485091,2.0
9,HELOC_XAI_010,8777,0,1,0.504549,borderline positive,average_duration_of_resolution,increases model at-risk output,average_duration_of_resolution,increases local at-risk prediction,True,5,1.000000,1.00,False,no_counterfactual_returned_first_pass,NaN,NaN


## 10. Case-type comparison

The four fixed case types are summarized descriptively. No significance tests are applied, and unavailable DiCE means remain missing.

In [9]:
case_metrics=cross.merge(shap_sparse[["case_id","features_for_50_percent","features_for_80_percent"]].rename(columns=lambda x:"shap_"+x if x!="case_id" else x),on="case_id")
case_metrics=case_metrics.merge(lime_sparse[["case_id","features_for_50_percent","features_for_80_percent"]].rename(columns=lambda x:"lime_"+x if x!="case_id" else x),on="case_id")
case_type=case_metrics.groupby("case_type",sort=False).agg(case_count=("case_id","size"),mean_top_5_overlap_count=("top_5_overlap_count","mean"),mean_top_5_jaccard=("top_5_jaccard","mean"),mean_direction_agreement_rate=("direction_agreement_rate","mean"),mean_shap_features_for_50_percent=("shap_features_for_50_percent","mean"),mean_shap_features_for_80_percent=("shap_features_for_80_percent","mean"),mean_lime_features_for_50_percent=("lime_features_for_50_percent","mean"),mean_lime_features_for_80_percent=("lime_features_for_80_percent","mean"),dice_counterfactual_availability_rate=("dice_valid_counterfactual","mean"),mean_dice_changed_modifiable_features=("dice_changed_modifiable_features","mean")).reset_index()
assert case_type.case_count.eq(5).all(); case_type.to_csv(ARTIFACT_DIR/"xai_case_type_comparison.csv",index=False); case_type

,case_type,case_count,mean_top_5_overlap_count,mean_top_5_jaccard,mean_direction_agreement_rate,mean_shap_features_for_50_percent,mean_shap_features_for_80_percent,mean_lime_features_for_50_percent,mean_lime_features_for_80_percent,dice_counterfactual_availability_rate,mean_dice_changed_modifiable_features
0,high-confidence positive,5,3.6,0.571429,1.0,3.0,7.0,2.4,4.0,0.0,NaN
1,borderline positive,5,4.2,0.733333,1.0,3.0,6.4,2.0,4.0,0.6,2.666667
2,high-confidence negative,5,4.0,0.666667,0.8,3.0,6.0,2.6,4.0,0.0,NaN
3,borderline negative,5,3.8,0.638095,1.0,3.4,6.6,2.0,4.0,1.0,2.800000


## 11. Special-code disclosure

The raw source contains `-7`, `-8`, and `-9`, whose meanings could not be verified from the frozen Hugging Face metadata. Stage 03 retained separate indicators while imputing underlying measurements with training-only medians. SHAP and LIME did not invent meanings, and DiCE did not generate or modify these states.

## 12. Visual evaluation

In [10]:
plot_paths=[]
fig,ax=plt.subplots(figsize=(9,4)); ax.bar(agreement.case_id,agreement.top_5_overlap_count,color="#4C78A8"); ax.set_ylabel("Shared top-five features"); ax.set_ylim(0,5); ax.tick_params(axis="x",rotation=75); fig.tight_layout(); path=PLOT_DIR/"heloc_xai_shap_lime_top5_agreement.png"; fig.savefig(path,dpi=180); plt.close(fig); plot_paths.append(path)
fig,ax=plt.subplots(figsize=(9,4)); ax.bar(direction.case_id,direction.direction_agreement_rate,color="#59A14F"); ax.set_ylabel("Direction agreement rate"); ax.set_ylim(0,1); ax.tick_params(axis="x",rotation=75); fig.tight_layout(); path=PLOT_DIR/"heloc_xai_direction_agreement.png"; fig.savefig(path,dpi=180); plt.close(fig); plot_paths.append(path)
fig,ax=plt.subplots(figsize=(7,4)); temp=sparsity_comparison.pivot(index="method",columns="metric",values="mean"); temp.plot.bar(ax=ax); ax.set_ylabel("Mean feature count"); ax.legend(fontsize=8); fig.tight_layout(); path=PLOT_DIR/"heloc_xai_sparsity_comparison.png"; fig.savefig(path,dpi=180); plt.close(fig); plot_paths.append(path)
fig,ax=plt.subplots(figsize=(6,4)); ax.bar(faithfulness_comparison.method,faithfulness_comparison.mean_absolute_score_change,color=["#F28E2B","#E15759"]); ax.set_ylabel("Mean absolute probability change"); fig.tight_layout(); path=PLOT_DIR/"heloc_xai_faithfulness_comparison.png"; fig.savefig(path,dpi=180); plt.close(fig); plot_paths.append(path)
fig,ax=plt.subplots(figsize=(7,4)); counts=pd.Series({"Valid counterfactual":len(dice_eval),"Completed no-CF":final_no_cf,"Technical error":final_technical}); ax.bar(counts.index,counts.values,color=["#59A14F","#BAB0AC","#E15759"]); ax.set_ylabel("Cases"); ax.set_ylim(0,20); fig.tight_layout(); path=PLOT_DIR/"heloc_xai_dice_outcomes.png"; fig.savefig(path,dpi=180); plt.close(fig); plot_paths.append(path)
[p.name for p in plot_paths]

['heloc_xai_shap_lime_top5_agreement.png',
 'heloc_xai_direction_agreement.png',
 'heloc_xai_sparsity_comparison.png',
 'heloc_xai_faithfulness_comparison.png',
 'heloc_xai_dice_outcomes.png']

## 13. Interpretation of Cross-Method XAI Evaluation

SHAP provides additive attribution aggregated to the 22 original features, with persisted perturbation and sensitivity diagnostics. LIME provides local surrogate explanations; feature agreement and same-seed reproduction are informative, but local fidelity varies across cases. DiCE adds a constrained counterfactual view using three modifiable financial-state variables, with availability, sparsity, and proximity reported alongside completed no-CF outcomes and the technical-retry disclosure. These results do not establish causality, fairness, compliance, creditworthiness, actionability, or deployment robustness.

## 14. Evaluation summary and metadata

In [11]:
summary_rows=[
{"category":"SHAP-LIME agreement","metric":"top_1_agreement_rate","method":"SHAP/LIME","value":agreement.top_1_agreement.mean(),"cases_evaluated":20,"interpretation":"Same leading original feature"},
{"category":"direction agreement","metric":"overall_direction_agreement_rate","method":"SHAP/LIME","value":overall_direction,"cases_evaluated":20,"interpretation":"Direction only, not magnitude"},
{"category":"SHAP sparsity","metric":"mean_features_for_80_percent","method":"SHAP","value":shap_sparse.features_for_80_percent.mean(),"cases_evaluated":20,"interpretation":"Features covering 80% attribution"},
{"category":"LIME sparsity","metric":"mean_features_for_80_percent","method":"LIME","value":lime_sparse.features_for_80_percent.mean(),"cases_evaluated":20,"interpretation":"Features covering 80% local weight"},
{"category":"DiCE sparsity","metric":"mean_changed_modifiable_features","method":"DiCE","value":dice_sparse.number_of_changed_modifiable_features.mean(),"cases_evaluated":len(dice_sparse),"interpretation":"Modifiable financial-state variables"},
{"category":"SHAP preliminary stability","metric":"mean_top_5_overlap","method":"SHAP","value":shap_stability.top_5_overlap_count.mean(),"cases_evaluated":5,"interpretation":"Deterministic local perturbation"},
{"category":"LIME reproducibility","metric":"exact_reproduction_rate","method":"LIME","value":lime_stability.exact_reproduction.mean(),"cases_evaluated":5,"interpretation":"Same-seed reproduction"},
{"category":"SHAP sensitivity","metric":"mean_absolute_score_change","method":"SHAP","value":shap_faith.absolute_score_change.mean(),"cases_evaluated":5,"interpretation":"Preliminary model sensitivity"},
{"category":"LIME sensitivity","metric":"mean_absolute_score_change","method":"LIME","value":lime_faith.absolute_score_change.mean(),"cases_evaluated":5,"interpretation":"Preliminary model sensitivity"},
{"category":"LIME local fidelity","metric":"mean_local_fidelity_r2","method":"LIME","value":lime_fidelity.local_fidelity_r2.mean(),"cases_evaluated":20,"interpretation":"Local surrogate fit"},
{"category":"DiCE availability","metric":"valid_counterfactual_rate","method":"DiCE","value":len(dice_eval)/20,"cases_evaluated":20,"interpretation":"Within constrained search"},
{"category":"DiCE proximity","metric":"mean_normalized_proximity","method":"DiCE","value":dice_eval.proximity_score.mean(),"cases_evaluated":len(dice_eval),"interpretation":"Normalized numerical closeness"}]
evaluation_summary=pd.DataFrame(summary_rows); evaluation_summary.to_csv(ARTIFACT_DIR/"xai_evaluation_summary.csv",index=False)
created=["xai_shap_lime_agreement.csv","xai_direction_agreement.csv","xai_sparsity_comparison.csv","xai_stability_comparison.csv","xai_faithfulness_comparison.csv","xai_lime_local_fidelity_summary.csv","xai_dice_evaluation.csv","xai_cross_method_case_summary.csv","xai_case_type_comparison.csv","xai_evaluation_summary.csv","xai_evaluation_metadata.json"]
metadata={"dataset":"mstz/heloc risk configuration","evaluation_case_count":len(cases),"shap_case_count":shap_top.case_id.nunique(),"lime_case_count":lime_top.case_id.nunique(),"dice_status_case_count":dice_status.case_id.nunique(),"valid_dice_counterfactual_count":len(dice_eval),"final_dice_no_cf_count":final_no_cf,"final_dice_technical_error_count":final_technical,"technical_retry_case_count":len(dice_technical),"special_code_semantics_verified":False,"artifacts_used":list(required),"artifacts_created":created,"plots_created":[p.name for p in plot_paths],"model_retrained":False,"preprocessor_refitted":False,"shap_regenerated":False,"lime_regenerated":False,"dice_regenerated":False}
(ARTIFACT_DIR/"xai_evaluation_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
evaluation_summary

,category,metric,method,value,cases_evaluated,interpretation
0,SHAP-LIME agreement,top_1_agreement_rate,SHAP/LIME,0.400000,20,Same leading original feature
1,direction agreement,overall_direction_agreement_rate,SHAP/LIME,0.948718,20,"Direction only, not magnitude"
2,SHAP sparsity,mean_features_for_80_percent,SHAP,6.500000,20,Features covering 80% attribution
3,LIME sparsity,mean_features_for_80_percent,LIME,4.000000,20,Features covering 80% local weight
4,DiCE sparsity,mean_changed_modifiable_features,DiCE,2.750000,8,Modifiable financial-state variables
5,SHAP preliminary stability,mean_top_5_overlap,SHAP,5.000000,5,Deterministic local perturbation
6,LIME reproducibility,exact_reproduction_rate,LIME,1.000000,5,Same-seed reproduction
7,SHAP sensitivity,mean_absolute_score_change,SHAP,0.077449,5,Preliminary model sensitivity
8,LIME sensitivity,mean_absolute_score_change,LIME,0.047352,5,Preliminary model sensitivity
9,LIME local fidelity,mean_local_fidelity_r2,LIME,0.416343,20,Local surrogate fit


## 15. Quality checks

In [12]:
assert len(cases)==20 and shap_top.case_id.nunique()==20 and lime_top.case_id.nunique()==20 and dice_status.case_id.nunique()==20
assert len(shap_top)==100 and len(lime_top)==100 and len(shap_stability)==5 and len(lime_stability)==5 and len(shap_faith)==5 and len(lime_faith)==5
assert len(dice_eval)==json.loads((ARTIFACT_DIR/"dice_metadata.json").read_text())["valid_counterfactual_count"]
assert len(dice_changes)==len(dice_eval)*3 and final_technical==0 and len(cross)==20 and cross.case_id.is_unique
print("XAI cases evaluated:",len(cases)); print("SHAP cases:",shap_top.case_id.nunique()); print("LIME cases:",lime_top.case_id.nunique()); print("DiCE status cases:",dice_status.case_id.nunique()); print("Valid DiCE counterfactuals:",len(dice_eval))
print("SHAP-LIME top-1 agreement:",f"{agreement.top_1_agreement.sum()}/20 ({agreement.top_1_agreement.mean():.1%})")
print("Mean SHAP-LIME top-3 overlap:",agreement.top_3_overlap_count.mean()); print("Mean SHAP-LIME top-3 Jaccard:",agreement.top_3_jaccard.mean()); print("Mean SHAP-LIME top-5 overlap:",agreement.top_5_overlap_count.mean()); print("Mean SHAP-LIME top-5 Jaccard:",agreement.top_5_jaccard.mean())
print("Overall directional agreement:",overall_direction)
print("SHAP preliminary stability mean overlap:",shap_stability.top_5_overlap_count.mean()); print("LIME same-seed reproducibility mean overlap:",np.nanmean(lime_overlap))
print("SHAP sensitivity mean score change:",shap_faith.absolute_score_change.mean()); print("LIME sensitivity mean score change:",lime_faith.absolute_score_change.mean())
print("LIME mean local fidelity R2:",lime_fidelity.local_fidelity_r2.mean()); print("LIME mean absolute local prediction error:",lime_fidelity.absolute_local_prediction_error.mean())
print("DiCE counterfactual availability:",len(dice_eval)/len(cases)); print("Mean changed modifiable features:",dice_sparse.number_of_changed_modifiable_features.mean()); print("Mean DiCE proximity score:",dice_eval.proximity_score.mean())
print("Final DiCE no-CF outcomes:",final_no_cf); print("Final DiCE technical errors:",final_technical); print("Controlled technical retry cases:",len(dice_technical))
print("Model retrained: False\nPreprocessor refitted: False\nSHAP regenerated: False\nLIME regenerated: False\nDiCE regenerated: False")

XAI cases evaluated: 20
SHAP cases: 20
LIME cases: 20
DiCE status cases: 20
Valid DiCE counterfactuals: 8
SHAP-LIME top-1 agreement: 8/20 (40.0%)
Mean SHAP-LIME top-3 overlap: 2.1
Mean SHAP-LIME top-3 Jaccard: 0.5599999999999999
Mean SHAP-LIME top-5 overlap: 3.9
Mean SHAP-LIME top-5 Jaccard: 0.6523809523809523
Overall directional agreement: 0.9487179487179487
SHAP preliminary stability mean overlap: 5.0
LIME same-seed reproducibility mean overlap: 5.0
SHAP sensitivity mean score change: 0.0774490418245849
LIME sensitivity mean score change: 0.047351510827270446
LIME mean local fidelity R2: 0.41634337694493073
LIME mean absolute local prediction error: 0.10420664950601441
DiCE counterfactual availability: 0.4
Mean changed modifiable features: 2.75
Mean DiCE proximity score: 0.1338568789584799
Final DiCE no-CF outcomes: 12
Final DiCE technical errors: 0
Controlled technical retry cases: 4
Model retrained: False
Preprocessor refitted: False
SHAP regenerated: False
LIME regenerated: False
